### Entity Extraction Approach

A hybrid entity extraction pipeline was used to identify meaningful counterparties and merchants from banking transaction narrations:

- **Tier 1 – Structural Parsing (Regex):** Extract entities from known transaction formats such as UPI, IMPS, and POS using predefined patterns.
- **Tier 2 – Merchant Dictionary Matching:** Identify known merchants and institutions (e.g., Amazon, Zomato, LIC, PhonePe) using keyword matching.
- **Tier 3 – Residual Text Extraction:** Remove banking keywords, transaction codes, and numeric identifiers; treat the remaining meaningful text as the likely entity name.
- **Fallback:** If no meaningful entity can be identified, classify the transaction entity as `Unknown`.

In [17]:
import pandas as pd
import re

In [18]:
df = pd.read_csv('Sample AA Data.csv')
# Drop empty narrations
df = df.dropna(subset=['NARRATION'])

In [19]:
def extract_entity(narration):
    narration = str(narration).upper()
    # TIER 1: PATTERN MATCHING (Regex)

    # Pattern A: "Payment from/to X" (Common in UPI)
    # Looks for "PAYMENT FROM" or "PAYMENT TO" and grabs the word(s) after it
    match = re.search(r'PAYMENT (?:FROM|TO)\s*([A-Z]+)', narration)
    if match: return match.group(1)
    # Pattern B: IMPS/NEFT Transfers with slashes
    # Example: IMPS/427419033405/FIN-XX811-SURESH/IMPS
    if 'IMPS/' in narration or 'NEFT/' in narration:
        parts = narration.split('/')
        if len(parts) >= 3:
            # The entity is usually in the 3rd section, often after a dash
            entity_section = parts[2]
            # If it looks like FIN-XX811-SURESH, grab the last part
            if '-' in entity_section:
                return entity_section.split('-')[-1]
            return entity_section
    # Pattern C: POS (Point of Sale) Swipes
    # Example: POS 45678123 RELIANCE FRESH DEL
    if 'POS ' in narration:
        # Remove "POS" and numbers, keep the alphabetical words
        clean_pos = re.sub(r'POS\s*\d+', '', narration).strip()
        words = clean_pos.split()
        if len(words) > 0:
            # Return the first 1-2 words as the merchant
            return " ".join(words[:2])

    #TIER 2: KNOWN DICTIONARY LOOKUP
    known_merchants = [
        'ZOMATO', 'SWIGGY', 'AMAZON', 'FLIPKART', 'UBER', 'OLA',
        'LIC', 'ZERODHA', 'GROWW', 'PHONEPE', 'PAYTM', 'CRED', 'BAJAJ'
    ]
    for merchant in known_merchants:
        if merchant in narration:
            return merchant

    # TIER 3:CLEANER
    # If no patterns matched, let's remove common banking jargon and numbers
    # to see what is left behind.
    jargon = ['UPI', 'IMPS', 'NEFT', 'RTGS', 'POS', 'DEP', 'TFR', 'INB', 'ACH', 'DR', 'CR']

    # Remove numbers and special characters
    clean_text = re.sub(r'[^A-Z\s]', ' ', narration)

    # Remove jargon words
    words = clean_text.split()
    entity_words = [w for w in words if w not in jargon and len(w) > 2]

    if entity_words:
        return " ".join(entity_words[:2]) # Return the first two meaningful words

    return "UNKNOWN ENTITY"



In [22]:
def label_entity_role(row):
    if str(row['TYPE']).upper() == 'CREDIT':
        return 'Payer (Source)'
    else:
        return 'Payee (Recipient)'

df['ENTITY_ROLE'] = df.apply(label_entity_role, axis=1)

In [25]:
df['EXTRACTED_ENTITY'] = df['NARRATION'].apply(extract_entity)
print("\n  RESULTS ")
df[['NARRATION', 'EXTRACTED_ENTITY','ENTITY_ROLE']].head(10)
# df.to_csv('Entity_Extracted_Data.csv', index=False)


  RESULTS 


,NARRATION,EXTRACTED_ENTITY,ENTITY_ROLE
0,UPI 416452325438Payment from PhonePe,PHONEPE,Payee (Recipient)
1,UPI 419523762861Payment from PhonePe,PHONEPE,Payee (Recipient)
2,DEP TFR INB IMPS/427419033405/FIN-XX811-Sure...,SURESH,Payer (Source)
3,UPI 440125342280Payment from PhonePe,PHONEPE,Payee (Recipient)
4,UPI 406516147581Payment from PhonePe,PHONEPE,Payee (Recipient)
5,UPI 446083786320Payment from PhonePe,PHONEPE,Payee (Recipient)
7,UPI 444596412300Payment from PhonePe,PHONEPE,Payee (Recipient)
8,DEP TFR UPI/CR/274316033324/BAL KISH/SBIN/XXX/,BAL KISH,Payer (Source)
9,UPI 410028523974Payment from PhonePe,PHONEPE,Payer (Source)
10,UPI 456869237946Payment from PhonePe,PHONEPE,Payer (Source)


In [29]:
df['EXTRACTED_ENTITY'].value_counts()

,count
EXTRACTED_ENTITY,
PHONEPE,63
UNKNOWN ENTITY,4
XXXXXX,4
MONEYTRF TXN,2
SURESH,2
MAHESHPU UCBA,2
PAYTM,2
MAHESHPU BKID,1
BAL KISH,1
